# PURE-GNN v3.1 HISTORICAL-COMPATIBLE SCIENTIFIC SCREEN
# STATUS: SOURCE REVIEW REQUIRED
# SCIENTIFIC TRAINING DISABLED

This notebook orchestrates the historical-compatible scientific screening protocol for **Pure-GNN v3.1**.

### Environment Setup Requirements:
- Accelerator: NVIDIA Tesla T4 GPU
- Internet: ON (for package setup and commit verification)
- Kaggle Input: `doduyquynii/fer13-split`

### Data Roles:
- `train.csv` (28,709 rows): Official training set.
- `val.csv` (3,589 rows): Official validation and checkpoint selection set.
- Holdout test set: Strictly forbidden from filesystem path configuration, reading, or evaluation.


## 1. User & Source Lock Configuration

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Irthn1311/FER2013_Graph.git"
REPO_BRANCH = "research/pure-gnn-v31"

# HARD SOURCE LOCK: Must reference an immutable tag after review passes.
# Must remain None until independent reviewer assigns the reviewed tag.
REVIEWED_SOURCE_TAG = None

# Kaggle Dataset Paths (Explicit Train and Validation ONLY)
FER_INPUT_ROOT = Path("/kaggle/input/datasets/doduyquynii/fer13-split/fer13-split")
TRAIN_CSV_PATH = FER_INPUT_ROOT / "train.csv"
VAL_CSV_PATH = FER_INPUT_ROOT / "val.csv"

OUTPUT_ROOT = Path("/kaggle/working/outputs/pure_gnn_v31/scientific_screen")
PACKAGE_RELATIVE = Path("research/pure_gnn_v31")

# Execution Switches
RUN_TESTS = True
RUN_DATA_VALIDATION = True
# HARD EXECUTION GATE: Must remain False. Scientific training is NOT authorized.
RUN_SCIENTIFIC_SCREEN = False

CONDITIONS = ["G0", "G0.5", "G1"]
PRIMARY_COMPARISON = "G1 - G0.5"
CANONICAL_SEED = 42

print("Configured Pure-GNN v3.1 Scientific Screen Runner:")
print(f"  Reviewed Source Tag: {REVIEWED_SOURCE_TAG}")
print(f"  Run Tests: {RUN_TESTS}")
print(f"  Run Data Validation: {RUN_DATA_VALIDATION}")
print(f"  Run Scientific Screen: {RUN_SCIENTIFIC_SCREEN}")


## 2. Immutable Source Validation

In [ ]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

WORKING = Path("/kaggle/working")
PROJECT_PATH = WORKING / "FER2013_Graph"

def run_checked(command, cwd=None, capture=False):
    actual = [str(item) for item in command]
    display = [re.sub(r"(https://x-access-token:)[^@]+@", r"\1***@", item) for item in actual]
    print("$", " ".join(display))
    result = subprocess.run(
        actual, cwd=cwd, text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )
    if result.returncode:
        if capture and result.stdout:
            print("\n".join(result.stdout.splitlines()[-100:]))
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result.stdout if capture else ""

if REVIEWED_SOURCE_TAG is None:
    print("NOTICE: REVIEWED_SOURCE_TAG is None. Running in pre-authorization audit mode.")
else:
    if not PROJECT_PATH.exists():
        run_checked(["git", "clone", "--branch", REVIEWED_SOURCE_TAG, "--single-branch", REPO_URL, PROJECT_PATH])
        os.chdir(PROJECT_PATH)
    elif PROJECT_PATH.exists():
        os.chdir(PROJECT_PATH)


## 3. Environment Inspection

In [ ]:
import platform
import tensorflow as tf

print("Python:", sys.version)
print("Platform:", platform.platform())
print("TensorFlow:", tf.__version__)
print("Physical Devices:", tf.config.list_physical_devices())
print("GPUs:", tf.config.list_physical_devices("GPU"))


## 4. Package Installation & Import Isolation

In [ ]:
PACKAGE_PATH = Path(os.getcwd()) / PACKAGE_RELATIVE
if PACKAGE_PATH.is_dir():
    run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", str(PACKAGE_PATH), "--no-deps"])
    import pure_gnn_v31
    print("Imported pure_gnn_v31 from:", pure_gnn_v31.__file__)
else:
    print("Package path not yet available; skipping editable install.")


## 5. Bounded Test Suite

In [ ]:
if RUN_TESTS and PACKAGE_PATH.is_dir():
    test_output = run_checked([sys.executable, "-m", "pytest", str(PACKAGE_PATH / "tests"), "-q"], capture=True)
    print("\n".join(test_output.splitlines()[-25:]))
else:
    print("Skipping tests.")


## 6. Dataset Row Count & Schema Validation

In [ ]:
import csv
import hashlib

if RUN_DATA_VALIDATION:
    for name, path, expected_rows in [("train", TRAIN_CSV_PATH, 28709), ("val", VAL_CSV_PATH, 3589)]:
        if not path.is_file():
            raise FileNotFoundError(f"{name.upper()} dataset file not found at expected path: {path}")
        sha = hashlib.sha256(path.read_bytes()).hexdigest()
        with path.open("r", encoding="utf-8", newline="") as f:
            reader = csv.reader(f)
            header = [col.strip().lower() for col in next(reader)]
            count = sum(1 for _ in reader)
        assert "emotion" in header and "pixels" in header, f"Invalid header in {name}: {header}"
        assert count == expected_rows, f"{name} row count mismatch: {count} != {expected_rows}"
        print(f"{name.upper()} Dataset Validated: rows={count}, sha256={sha}")


## 7. Scientific Screen Execution (FAIL-CLOSED GATE)

In [ ]:
if RUN_SCIENTIFIC_SCREEN:
    raise PermissionError(
        "Scientific execution has not passed independent source review."
    )
else:
    print("Scientific training is disabled (RUN_SCIENTIFIC_SCREEN=False).")
    print("Ready for independent source review.")
